# Fp1 – Friedel Pair Matching

*Written by Jean-Baptiste Jacob, May 2026*

Match **Friedel pairs** in a large scanning-3DXRd dataset using the`friedel_pairs` module from ImageD11. 

For batch processing a list of datasets, use this notebook to test which parameters work the best for the search, before running the batch process (fp1b_run_match_friedel_pairs)

For details about friedel pairs matching algorithm see `match_friedel_pairs.ipynb` notebook in ImageD11/nbGui

### Load data

In [ ]:
import os, sys, time
start = time.time()

# python environment stuff
IMAGED11_PATH = '/home/esrf/jean1994b/ImageD11_jbjacob'  # None means do not use git, otherwise enter the name of the folder to use for the git checkout "ImageD11" or "ImageD11_version_xx", etc
CHECKOUT_PATH = 'ImageD11'  # the name of the git checkout folder within path. None means guess


if IMAGED11_PATH is not None:
    if '/data/id11/nanoscope' not in sys.path:
        sys.path.append('/data/id11/nanoscope')
    import install_ImageD11_from_git
    install_ImageD11_from_git.setup_ImageD11_from_git(IMAGED11_PATH,CHECKOUT_PATH)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ImageD11.sinograms.dataset
import ImageD11.sinograms.geometry
import ImageD11.friedel_pairs as fp
import logging
import copy

from pf3dxrd.pf3dxrd import utils

logging.basicConfig(level=logging.INFO)

%load_ext autoreload
%autoreload 2
%matplotlib ipympl

In [ ]:
# get dataset file
dsfile = 'MgO_3_0p1M_12um_0003/MgO_3_0p1M_12um_0003_dataset.h5'

In [ ]:
ds = ImageD11.sinograms.dataset.load(dsfile)
print(ds)

In [ ]:
# we will index directly from the 2d peaks. No need to use the 2D to 4D peaks mapping as with the tomo route 
cf_2d = ds.get_cf_2d()
cf_2d.parameters.loadparameters(ds.parfile)
cf_2d.updateGeometry()
print(cf_2d.nrows)

In [ ]:
# plot some satistics about 2D peaks
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].plot(cf_2d.ds[::10], cf_2d.eta[::10], ',')
axes[0].set_title('d* vs eta')

axes[1].hist(cf_2d.Number_of_pixels, 100)
axes[1].set_yscale('log')
axes[1].set_title(' Number_of_pixels')

axes[2].hist(np.log10(cf_2d.sum_intensity), 100)
axes[2].set_title('log(sum_intensity)')

plt.tight_layout()


In [ ]:
# Optional: you may want to filter out some peaks (weak peaks, peaks with laege d-star in the corner of the detector,...)
cf_2d.filter(cf_2d.sum_intensity>10)
cf_2d.filter(cf_2d.Number_of_pixels>=4)
cf_2d.filter(cf_2d.ds < 1.6)
cf_2d.nrows

### Chunk-based pairing

For large scanning-3DXRD datasets (>10M peaks), searching pairs within the full columnfile is extremely inefficient.
Instead, the columnfile is divided into small **mirror chunks** — pairs of mirror subsets that potentially contain the Friedel mates of one another — and matching is done independently within each chunk, then merged. This approch is much more memory efficient as global pairing for large datasets and can be easily paralellized, as each pair of chunks can be treated independently.

#### Initialize FriedelPairIndexer instance

In [ ]:
# copy here the y0 guess from the previous notebook
y0=11.225555555555555

In [ ]:
# Initiate a new FriedelPairsIndexer instance with 2d peaks
FPIndexer = fp.FriedelPairIndexer(cf_2d, copy.deepcopy(ds),
                                   tol_gv=0.08,
                                   tol_eta=0.4,
                                   tol_logI=0.4,
                                   n_steps=20)

In [ ]:
# set PeakSubsets instance. If y0 is None, it is automatically set to the value of the central dty bin in ds. 
# n_eta_bins: needed to match eta pairs.
FPIndexer.set_peak_subsets(n_eta_bins=360, y0 = y0)
FPIndexer.sort_peak_subsets(pair_type='omega')

In [ ]:
# check symmetry: the blue and orange plot should roughly overlap: same number of peaks / total intensity in paired subsets. If not, there is something wrong with y0
# If the shift is less than half y-step, this does not matter much
fig = FPIndexer.PeakSubsets.check_symmetry()

#### Match friedel pairs (omega pairs)

`match_friedel_pairs_by_chunks()` orchestrates the full pipeline:

1. **Initialisation** — builds `PeakSubsets`, computes valid chunk pairs (both members must contain at least one peak)
2. **Pilot pairing** — picks a representative chunk near the centre of the sinogram and runs a quick search to auto-calibrate the search-space weights
3. **Parallel matching** — dispatches all valid chunk pairs to a pool of worker processes; each worker calls `_run_pairing` silently and returns a local result dict
4. **Merging** — local pair IDs from each chunk are offset to be globally unique and written into the `omega_pair_id` column of the master columnfile

**Notes**
- `chunk_type`    : str `frames`, `scans`, `eta_bins` ( default: `scans`)
                For omega pairs, use `scans` (recommended) or `frames` for friedel pair search at different levels of chunking granularity.
                For eta pairs, use `eta_bins`. Granularity is adjusted with `n_eta_bins` in `set_peak_subsets`
- `extended_bin_search` : match pairs in overlaping blocks of 3 eta bins / 3 dty bins instead of single bins. better completeness but slower than the regular approach
- `drop_unpaired` : bool (default False). remove unpaired peaks from peakfile. 
- `reset_psub`    : bool (default False). reinitialize self.Peaksubset from scratch if True. 
- `n_workers`     : int number of parallel worker processes. -1 = use all available cores
- `filter_mode`   : str; 'strict' or 'relaxed' (default strict). `strict` should give better completeness but is slower
- Set `n_workers=-1` to use all available cores. Use `n_workers=1` for serial execution (easier to debug).


In [ ]:
# Omega pairs. use 'scans' subset, it is usually much faster than 'frames' subsets 
cf_paired = FPIndexer.match_friedel_pairs_by_chunks(chunk_type='scans',
                                                    extended_bin_search=True,
                                                    filter_mode='relaxed',
                                                    drop_unpaired=False,
                                                    reset_psub=False,
                                                    doplot=True,
                                                    n_workers=-1)

#### Match eta pairs (optional)
not needed for the rest of the fp route, but can be done for completeness. 

In [ ]:
# sort peakfile by eta bisn and check symmetry, as previously done for the omega pairs
FPIndexer.sort_peak_subsets(pair_type='eta_bins')
FPIndexer.PeakSubsets.check_symmetry()

In [ ]:
# match eta pairs
cf_paired = FPIndexer.match_friedel_pairs_by_chunks(chunk_type='eta_bins',
                                                    extended_bin_search=False,
                                                    filter_mode='relaxed',
                                                    drop_unpaired=False,
                                                    reset_psub=False,
                                                    doplot=True,
                                                    n_workers=-1)

In [ ]:
# pair distance in two-theta. 
io1, io2 = fp.get_pairs(cf_2d, 'omega')
dtth_om = cf_2d.tth[io1] - cf_2d.tth[io2]

# If eta pairs were not matched, comment these two lines
ie1, ie2 = fp.get_pairs(cf_2d, 'eta')
dtth_eta = cf_2d.tth[ie1] - cf_2d.tth[ie2]


plt.figure()
plt.hist(dtth_om,100, label='omega_pairs');
plt.hist(dtth_eta,100,alpha=.6, label='eta_pairs'); # comment if eta pairs not matched
plt.title('two-theta distance between pairs')
plt.xlabel('delta_tth (deg)')
plt.legend()

### Visualize outputs

#### reconstruction in sample space
Reconstruct an image of the sample by mapping the total intensity of diffraction peaks reprojected in sample space.

In [ ]:
# histogram omega pairs
ds.y0=y0
sx, sy  = fp.locate_omega_pairs(cf_paired, (io1,io2), ds=ds, y0=ds.y0)
hist_guess = np.histogram2d(sx+ds.y0, sy+ds.y0, bins=ds.ybinedges)[0]

In [ ]:
# histogram eta pairs
sx_, sy_  = fp.locate_eta_pairs(cf_paired, (ie1,ie2), ds=ds, y0=ds.y0)
hist_guess_ = np.histogram2d(sx_+ds.y0, sy_+ds.y0, bins=ds.ybinedges)[0]

In [ ]:
fig, ax = plt.subplots(1,2,layout='constrained', figsize=(12,6))
ax1,ax2 = ax.ravel()

im1=ax1.pcolormesh(ds.ybinedges, ds.ybinedges, hist_guess,
                   vmin=np.percentile(hist_guess.flatten(),20),
                   vmax=np.percentile(hist_guess.flatten(),99.5))
ax1.set_aspect(1)
ax1.set(title=f'omega_pairs', xlabel='Sample X axis -->', ylabel='Sample Y axis -->')
cbar = plt.colorbar(im1, ax=ax1, orientation='vertical', pad=0.04, shrink=0.7)

# comment this block if eta pairs not matched
im2=ax2.pcolormesh(ds.ybinedges, ds.ybinedges, hist_guess_,
                   vmin=np.percentile(hist_guess_.flatten(),35),
                   vmax=np.percentile(hist_guess_.flatten(),99.5))
ax2.set_aspect(1)
ax2.set(title=f'eta_pairs', xlabel='Sample X axis -->', ylabel='Sample Y axis -->')
cbar = plt.colorbar(im2, ax=ax2, orientation='vertical', pad=0.04, shrink=0.7)

### Geometry correction
2θ angle and g-vector positions corrected from rotation center offset

In [ ]:
# cf copy with uncorrected peak positions
tth_raw = cf_2d.tth.copy()

In [ ]:
cf_2d.addcolumn(tth_raw, 'tthraw')

In [ ]:
# update geometry : recomputes tth, ds, gx, etc. using omega pairs relationships
fp.update_geometry_fpairs(cf_2d, ds)

Usually looks better on an histogram

In [ ]:
def get_histogram(cf, tthcol='tth', tthrange=(0,20), tth_step = 0.005, mask=None):
    
    if mask is None:
        m = np.full(cf.nrows, True)
    else:
        m = mask
    
    # select tth col + range
    tth = cf.getcolumn(tthcol)
    msk = np.all([tth <= tthrange[1], tth >= tthrange[0], m], axis=0)
    print(msk.sum())
    tth_sel = tth[msk]
    
    hist, binedges = np.histogram(tth_sel, bins=np.arange(tthrange[0], tthrange[1], tth_step))
    bincens = binedges[1:] - tth_step / 2
    
    return hist, bincens, binedges

In [ ]:
# plot 2-theta vs. eta for corrected vs non-corrected tth on a subset of the two-theta range
tthrange = (2,16)
m = cf_2d.omega_pair_id > -1

fig = plt.figure(figsize=(10,5))
# add histogram: better visualization of tth position
fig.add_subplot(111)
h,b,_ = get_histogram(cf_2d, 'tthraw', tthrange, tth_step = 0.002, mask=None)
hc,bc,_ = get_histogram(cf_2d, 'tth', tthrange, tth_step = 0.002, mask=m)

plt.plot(b,h,'-', lw=.6, label='non-corrected 2-theta')
plt.plot(bc,hc,'-',lw=.6, label='corrected 2-theta')
plt.legend()
plt.xlim(tthrange)
plt.xlabel('2-theta (deg)')
plt.ylabel('counts')

#### Save peakfile

**Note on saving**: the colfile_to_hdf function in ImageD11.columnfile saves every column in 64b (either floats or ints), which can quickly make very big files and is not very efficient. 
I usually use an alternative function in pf3dxrd.utils (colf_to_hdf), which tries to optimize this a little better using shortints (1-b), ints (32b) and longints (64b) depending on what is stored in each column. 
If saving mode is 'minimal', it will save only the necessary columns, than cannot be recomputed from the other ones, and drop everything else (tth, gx, ds, etc.). 



In [ ]:
# optional: filter unpaired peaks
paired = (cf_2d.omega_pair_id>-1) | (cf_2d.eta_pair_id>-1)
cf_2d.filter(paired)
cf_2d.nrows

In [ ]:
# save to hdf5
colfile = ds.col2dfile.replace('peaks_2d','peaks_2d_paired')  # write paired peaks in a separate columnfile *peaks_2d_paired
utils.colf_to_hdf(cf_2d, colfile)